# 36. Neural Networks: Generative Adversarial Networks (GANs)

## Algorithm Category
**Type**: Neural Networks - Generative Models  
**Complexity**: Very High  
**Use Case**: Image generation, data augmentation, unsupervised learning

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand GAN architecture and adversarial training
- Implement a simple GAN from scratch
- Understand generator and discriminator networks
- Train GANs and visualize generated samples
- Understand common GAN challenges and solutions

## Historical Context

GANs were introduced by Goodfellow et al. in 2014:
- Goodfellow, I., et al. (2014): "Generative Adversarial Nets"
- Revolutionary approach to generative modeling
- Foundation for modern image generation

**Key Papers/References:**
- Goodfellow, I., et al. (2014). "Generative Adversarial Nets"
- Radford, A., et al. (2015). "Unsupervised representation learning with deep convolutional generative adversarial networks"

## When to Use GANs

GANs are appropriate when:
- Need to generate realistic data
- Image generation and synthesis
- Data augmentation
- Unsupervised learning
- Style transfer
- When you have large datasets

## Theory & Mechanics

### Mathematical Foundation

**GAN Objective (Minimax Game):**
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]$$

Where:
- $G$: Generator network
- $D$: Discriminator network
- $p_{data}$: Real data distribution
- $p_z$: Noise distribution (latent space)

**Generator Loss:**
$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z(z)}[\log D(G(z))]$$

**Discriminator Loss:**
$$\mathcal{L}_D = -\mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]$$

### Key Components

1. **Generator (G)**
   - Takes random noise as input
   - Generates fake data
   - Tries to fool discriminator
   - Learns data distribution

2. **Discriminator (D)**
   - Classifies real vs fake
   - Tries to distinguish real from generated
   - Provides feedback to generator

3. **Adversarial Training**
   - Two networks compete
   - Generator improves to fool discriminator
   - Discriminator improves to detect fakes
   - Equilibrium when generator produces realistic data

### How It Works

1. **Initialize**: Random generator and discriminator
2. **Train Discriminator**: On real and fake data
3. **Train Generator**: To fool discriminator
4. **Alternate**: Between training D and G
5. **Converge**: When generator produces realistic data

### Key Hyperparameters

- **latent_dim**: Dimension of noise vector
- **learning_rate**: Step size (often different for G and D)
- **batch_size**: Number of samples per batch
- **n_critic**: Number of D updates per G update
- **beta1, beta2**: Adam optimizer parameters

### Advantages

- Can generate realistic data
- Unsupervised learning
- No explicit likelihood needed
- Produces diverse samples
- State-of-the-art image generation

### Limitations

- Training instability
- Mode collapse (limited diversity)
- Hard to evaluate
- Requires careful tuning
- Computationally expensive


## Implementation

Let's implement a simple GAN for generating 2D data.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# PyTorch: Deep learning framework
import torch  # PyTorch core library (tensors, automatic differentiation)
import torch.nn as nn  # Neural network layers and modules
import torch.optim as optim  # Optimization algorithms (Adam, SGD, etc.)
from torch.utils.data import DataLoader, TensorDataset  # Data loading utilities

# ============================================
# GPU DETECTION: Using CUDA for Faster Training
# ============================================

# Check if CUDA (GPU) is available
# GANs are computationally intensive - GPU acceleration is highly recommended
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# torch.cuda.is_available(): Returns True if GPU is available
# If GPU available: device = 'cuda' (use GPU)
# If no GPU: device = 'cpu' (use CPU - much slower for GANs)
print(f"Using device: {device}")  # Display which device we're using

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING SYNTHETIC DATA: 2D Circle Distribution
# ============================================

# GANs learn to generate data that matches a given distribution
# We'll use a simple 2D circle distribution as our "real" data
# The generator will learn to create points that form a circle

def generate_circle_data(n_samples=1000):
    """
    Generate 2D data points forming a circle with noise.
    
    Args:
        n_samples: Number of data points to generate
    
    Returns:
        Array of shape (n_samples, 2) - (x, y) coordinates
    """
    # Generate random angles uniformly distributed around circle
    angles = np.random.uniform(0, 2*np.pi, n_samples)
    # np.random.uniform(): Random angles from 0 to 2π
    # This ensures points are evenly distributed around the circle
    
    # Set circle radius
    radius = 1.0  # Unit circle (radius = 1)
    
    # Convert angles to (x, y) coordinates
    # Add small random noise to make it more realistic
    x = radius * np.cos(angles) + np.random.normal(0, 0.1, n_samples)
    # radius * np.cos(angles): X coordinate on circle
    # np.random.normal(0, 0.1, n_samples): Small random noise (std=0.1)
    # Noise makes data more realistic (not perfectly circular)
    
    y = radius * np.sin(angles) + np.random.normal(0, 0.1, n_samples)
    # radius * np.sin(angles): Y coordinate on circle
    # Same noise added to Y coordinates
    
    # Stack X and Y into 2D array
    return np.column_stack([x, y])
    # Returns: (n_samples, 2) - each row is (x, y) coordinate

# ============================================
# GENERATING AND NORMALIZING REAL DATA
# ============================================

# Generate real data (the distribution we want GAN to learn)
real_data = generate_circle_data(n_samples=1000)
# Generate 1000 points forming a circle
# This is our "real" data distribution

# Normalize data (zero mean, unit variance)
# This helps neural networks train faster and more stably
real_data = (real_data - real_data.mean(axis=0)) / real_data.std(axis=0)
# real_data.mean(axis=0): Mean of each column (x and y)
# real_data.std(axis=0): Standard deviation of each column
# Formula: (x - mean) / std
# Result: Data centered at (0, 0) with std=1

# ============================================
# DISPLAYING DATA INFORMATION
# ============================================

print(f"Real data shape: {real_data.shape}")  # Should be (1000, 2)

# ============================================
# VISUALIZING REAL DATA DISTRIBUTION
# ============================================

# Plot real data to see the circle distribution
plt.figure(figsize=(6, 6))  # Square figure (6×6 inches)
plt.scatter(real_data[:, 0], real_data[:, 1], alpha=0.5, s=20)
# real_data[:, 0]: X coordinates
# real_data[:, 1]: Y coordinates
# alpha=0.5: Semi-transparent points (easier to see density)
# s=20: Point size

plt.title('Real Data (Circle)')  # Chart title
plt.xlabel('X')  # X-axis label
plt.ylabel('Y')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid
plt.axis('equal')  # Equal aspect ratio (circle looks like circle, not ellipse)
plt.tight_layout()  # Adjust layout
plt.show()  # Display the plot

# Interpretation:
# - Real data forms a circular distribution
# - GAN generator will learn to generate points that match this distribution
# - Discriminator will learn to distinguish real (circle) from fake (random) points
# - After training, generator should produce points forming a circle


In [ ]:
# ============================================
# DEFINING GENERATOR NETWORK: Creating Fake Data
# ============================================

# Generator: Takes random noise and generates fake data
# Goal: Generate data that looks like real data (fool the discriminator)
class Generator(nn.Module):
    """
    Generator network for GAN.
    
    Architecture:
    - Input: Random noise (latent_dim)
    - Output: Fake data (output_dim)
    - Learns to map noise to realistic data
    """
    
    def __init__(self, latent_dim=2, output_dim=2, hidden_dim=64):
        """
        Initialize Generator.
        
        Args:
            latent_dim: Dimension of input noise vector (2 for 2D noise)
            output_dim: Dimension of output data (2 for 2D points)
            hidden_dim: Number of hidden units (64)
        """
        super(Generator, self).__init__()
        # super() calls parent class (nn.Module) constructor
        
        # Sequential network: Stack of layers
        self.net = nn.Sequential(
            # Layer 1: Noise -> Hidden
            nn.Linear(latent_dim, hidden_dim),
            # Input: (batch, latent_dim) - random noise
            # Output: (batch, hidden_dim) - hidden representation
            # This expands the noise into a richer representation
            
            nn.ReLU(),  # ReLU activation (introduces non-linearity)
            
            # Layer 2: Hidden -> Hidden
            nn.Linear(hidden_dim, hidden_dim),
            # Another hidden layer (increases model capacity)
            
            nn.ReLU(),  # ReLU activation
            
            # Layer 3: Hidden -> Output
            nn.Linear(hidden_dim, output_dim),
            # Output: (batch, output_dim) - generated data points
            
            # Tanh activation: Output in [-1, 1]
            nn.Tanh()
            # Tanh ensures output is bounded (good for normalized data)
            # Range: [-1, 1] (matches normalized real data)
        )
    
    def forward(self, z):
        """
        Forward pass: Generate fake data from noise.
        
        Args:
            z: Random noise tensor (batch_size, latent_dim)
        
        Returns:
            Generated data tensor (batch_size, output_dim)
        """
        return self.net(z)
        # Input: Random noise
        # Output: Generated data points
        # Generator learns: "How to transform random noise into realistic data"

# ============================================
# DEFINING DISCRIMINATOR NETWORK: Detecting Fake Data
# ============================================

# Discriminator: Classifies real vs fake data
# Goal: Correctly identify real data and reject fake data
class Discriminator(nn.Module):
    """
    Discriminator network for GAN.
    
    Architecture:
    - Input: Data point (real or fake)
    - Output: Probability that input is real (0 to 1)
    - Learns to distinguish real from fake
    """
    
    def __init__(self, input_dim=2, hidden_dim=64):
        """
        Initialize Discriminator.
        
        Args:
            input_dim: Dimension of input data (2 for 2D points)
            hidden_dim: Number of hidden units (64)
        """
        super(Discriminator, self).__init__()
        
        # Sequential network: Stack of layers
        self.net = nn.Sequential(
            # Layer 1: Input -> Hidden
            nn.Linear(input_dim, hidden_dim),
            # Input: (batch, input_dim) - data point (real or fake)
            # Output: (batch, hidden_dim) - hidden representation
            
            # LeakyReLU: Better than ReLU for discriminators
            nn.LeakyReLU(0.2),
            # LeakyReLU(0.2): Negative values become 0.2 * x (not zero)
            # Prevents "dying ReLU" problem (neurons that never activate)
            # Common in GAN discriminators
            
            # Layer 2: Hidden -> Hidden
            nn.Linear(hidden_dim, hidden_dim),
            # Another hidden layer
            
            nn.LeakyReLU(0.2),  # LeakyReLU activation
            
            # Layer 3: Hidden -> Output
            nn.Linear(hidden_dim, 1),
            # Output: (batch, 1) - single score per data point
            
            # Sigmoid: Output probability in [0, 1]
            nn.Sigmoid()
            # Sigmoid converts score to probability
            # 0 = definitely fake, 1 = definitely real
            # 0.5 = uncertain
        )
    
    def forward(self, x):
        """
        Forward pass: Classify data as real or fake.
        
        Args:
            x: Data tensor (batch_size, input_dim) - real or fake data
        
        Returns:
            Probability tensor (batch_size, 1) - probability of being real
        """
        return self.net(x)
        # Input: Data point (real or fake)
        # Output: Probability that input is real (0 to 1)
        # Discriminator learns: "Is this data real or fake?"

print("Generator and Discriminator models defined!")  # Confirm models are ready

# Key Concepts:
# - Generator: Creates fake data from noise (tries to fool discriminator)
# - Discriminator: Classifies real vs fake (tries to catch generator)
# - Adversarial training: They compete, both improve over time
# - Equilibrium: When generator produces realistic data and discriminator can't tell the difference


In [ ]:
# Initialize models
latent_dim = 2
generator = Generator(latent_dim=latent_dim, output_dim=2, hidden_dim=64).to(device)
discriminator = Discriminator(input_dim=2, hidden_dim=64).to(device)

# Loss and optimizers
criterion = nn.BCELoss()
lr = 0.0002
beta1 = 0.5
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

# Prepare real data
real_data_tensor = torch.FloatTensor(real_data).to(device)
real_labels = torch.ones(len(real_data), 1).to(device)
fake_labels = torch.zeros(len(real_data), 1).to(device)

print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")


## Training

Let's train the GAN.


In [ ]:
# Training loop
num_epochs = 200
batch_size = 64
n_critic = 1  # Number of D updates per G update

G_losses = []
D_losses = []

for epoch in range(num_epochs):
    # Train Discriminator
    for _ in range(n_critic):
        # Real data
        idx = torch.randint(0, len(real_data), (batch_size,))
        real_batch = real_data_tensor[idx]
        real_label = real_labels[:batch_size]
        
        # Fake data
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_batch = generator(noise)
        fake_label = fake_labels[:batch_size]
        
        # Train D on real
        optimizer_D.zero_grad()
        D_real = discriminator(real_batch)
        loss_D_real = criterion(D_real, real_label)
        
        # Train D on fake
        D_fake = discriminator(fake_batch.detach())
        loss_D_fake = criterion(D_fake, fake_label)
        
        # Total D loss
        loss_D = (loss_D_real + loss_D_fake) / 2
        loss_D.backward()
        optimizer_D.step()
    
    # Train Generator
    optimizer_G.zero_grad()
    noise = torch.randn(batch_size, latent_dim).to(device)
    fake_batch = generator(noise)
    D_fake = discriminator(fake_batch)
    loss_G = criterion(D_fake, real_label)  # Try to fool D
    loss_G.backward()
    optimizer_G.step()
    
    # Save losses
    G_losses.append(loss_G.item())
    D_losses.append(loss_D.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: D_loss = {loss_D.item():.4f}, G_loss = {loss_G.item():.4f}")

print("\nTraining complete!")


## Visualization

Let's visualize the training progress and generated samples.


In [ ]:
# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(G_losses, label='Generator Loss', alpha=0.7)
plt.plot(D_losses, label='Discriminator Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Generate samples
generator.eval()
with torch.no_grad():
    noise = torch.randn(1000, latent_dim).to(device)
    generated_data = generator(noise).cpu().numpy()

# Visualize real vs generated
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].scatter(real_data[:, 0], real_data[:, 1], alpha=0.5, s=20, c='blue')
axes[0].set_title('Real Data')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

axes[1].scatter(generated_data[:, 0], generated_data[:, 1], alpha=0.5, s=20, c='red')
axes[1].set_title('Generated Data')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the GAN performance.


In [ ]:
# Check discriminator accuracy
discriminator.eval()
with torch.no_grad():
    # Real data
    D_real_pred = discriminator(real_data_tensor[:100])
    real_acc = (D_real_pred > 0.5).float().mean().item()
    
    # Generated data
    noise = torch.randn(100, latent_dim).to(device)
    fake_data = generator(noise)
    D_fake_pred = discriminator(fake_data)
    fake_acc = (D_fake_pred < 0.5).float().mean().item()

print("Discriminator Performance:")
print(f"  Real data accuracy: {real_acc:.3f}")
print(f"  Fake data accuracy: {fake_acc:.3f}")

# Assertions
assert len(G_losses) == num_epochs, "Should have trained for all epochs"
print("\n✓ Validation checks passed")

print("\nNote: GANs are notoriously difficult to train.")
print("Key challenges include mode collapse, training instability,")
print("and difficulty in evaluation.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **GAN Architecture**
   - Two competing networks: Generator and Discriminator
   - Adversarial training process
   - Minimax game formulation
   - No explicit likelihood needed

2. **Generator (G)**
   - Maps noise to data space
   - Learns to generate realistic samples
   - Tries to fool discriminator
   - Improves through adversarial feedback

3. **Discriminator (D)**
   - Classifies real vs fake
   - Provides training signal to generator
   - Improves to detect fakes
   - Should converge to 0.5 when G is perfect

4. **Training Challenges**
   - **Mode collapse**: Generator produces limited diversity
   - **Instability**: Hard to balance G and D
   - **Evaluation**: No clear metric for quality
   - **Convergence**: May not converge to Nash equilibrium

### When to Use GANs

✅ **Good for:**
- Image generation
- Data augmentation
- Unsupervised learning
- Style transfer
- When you need realistic samples
- Large datasets available

❌ **Not ideal for:**
- Small datasets
- When interpretability is needed
- Real-time applications
- When stability is critical
- When explicit likelihood is needed

### Next Steps

- Explore **DCGAN** (Deep Convolutional GAN) for images
- Try **WGAN** (Wasserstein GAN) for stability
- Use **Conditional GANs** for controlled generation
- Apply **Progressive GANs** for high-resolution images
- Experiment with **StyleGAN** for advanced image synthesis
